# 06 — Accessibility prediction: baseline classifier

Predicting whether an edge is likely wheelchair-accessible from its own attributes (highway type, surface, street_count), using the real tagged edges as labeled data. Baseline only — no graph-neighbor information yet (that's the GNN's job, next notebook).

### Download the area (reusing the cached data)

Same Downtown Toronto graph as notebooks 04/05.

In [ ]:
import sys
sys.path.append("..")

import osmnx as ox
from src.ml.features import build_feature_table, geographic_split, encode_features
from src.ml.baseline import train_baseline
from sklearn.metrics import accuracy_score, classification_report

ox.settings.cache_folder = "cache"
ox.settings.useful_tags_way += ["wheelchair", "surface"]
ox.settings.useful_tags_node += ["kerb", "crossing", "tactile_paving"]
ox.settings.overpass_url = "https://overpass.kumi.systems/api/interpreter"

G_big = ox.graph_from_place("Downtown Toronto, Toronto, Ontario, Canada", network_type="walk")
print(G_big.number_of_edges(), "edges")


### Collect labeled edges into a table

Only edges with a real `wheelchair` tag (`yes`/`designated` → 1, `no` → 0) become labeled rows — the other ~38,800 untagged edges are exactly what this model is meant to predict for later, so they can't be used as training labels now. Features per edge: `length`, `highway` type, `surface` (or `"unknown"` if missing), and `street_count` — which lives on the node, not the edge, so it's pulled from `G_big.nodes[u]`.

In [ ]:
df = build_feature_table(G_big, G_big.edges(keys=True, data=True))
print(df.shape)
print(df["label"].value_counts())
df.head()


### Geography-based train/test split

Not a random shuffle — split on each edge's longitude (`x` of node `u`) at the median, so train and test edges come from genuinely different halves of the area. This is a stricter, more honest test than a random split: it checks whether the model generalizes to a part of the city it never saw, not just to unseen edges that happen to be right next to ones it trained on.

In [ ]:
train_df, test_df = geographic_split(df, "x")
print("train:", len(train_df), " test:", len(test_df))

X_train, X_test = encode_features(train_df, test_df)
y_train, y_test = train_df["label"], test_df["label"]


### Train and evaluate the baseline

Same `RandomForestClassifier` pattern from the toy example, now on the real, geographically-split data.

In [ ]:
model = train_baseline(X_train, y_train)

predictions = model.predict(X_test)
print("accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))


In [5]:
print("train labels:")
print(train_df["label"].value_counts())
print()
print("test labels:")
print(test_df["label"].value_counts())

train labels:
label
1    77
0    10
Name: count, dtype: int64

test labels:
label
1    81
0     6
Name: count, dtype: int64
